In [ ]:
import google.generativeai as genai
# Get Google API Key from environment variable
api_key = os.environ["GOOGLE_API_KEY"]
genai.configure(api_key=api_key)
client = genai.GenerativeModel('gemini-pro-vision')

def generate(self, prompt, image, temperature=0.0):
    meta_prompt = "You are given a programming question, along with an image from the question statement. "
    "You are required to determine if the image is relevant to the question or not. "
    "For example, a formula or a figure of a data structure is relevant, but an image of fruit or a celebrity is not, if it does not contain any useful information for solving the question. "
    "You should give explanations for you judgement. \n"
    "If the image is relevant, output in a json block ```json\n{'relevance': True}```. Otherwise, output ```json\n{'relevance': False}``` \n"
    # Convert all images to base64

    interleaved_messages = [meta_prompt, image]
    try:
        response = self.client.generate_content(interleaved_messages, 
            generation_config=self.genai_package.types.GenerationConfig(
                temperature=temperature
            )
        )
        return response.text
    except Exception as e:
        print(e)
        try:
            print(response.prompt_feedback)
        except Exception as err:
            pass
        return None

def extract_result(self, response):
    pattern = r"```json(.*?)```"

    # response = response.split("###ANSWER:", maxsplit=1)[-1].strip()
    
    # Use re.DOTALL to make '.' match any character including a newline
    matches = re.findall(pattern, response, re.DOTALL)

    if matches:
        return matches[0]
    else:
        return None

In [ ]:
import json
from tqdm import tqdm
from utils import load_problems

problems = load_problems(args.problems_root)
            
for problem in tqdm(problems):
    for img_id, image in enumerate(problem["images"]):
        prompt = "![image](x)This image comes from a code contest problem. Please describe the image in details."
        hashnum = hash_pil_image(image)
        response = None
        attempts = 0
        # Try for up to 5 times to generate a response
        while response is None and attempts < 5:
            response = model.generate(prompt, [image], temperature=args.temperature)
            attempts += 1

        
        if not response:
            print(f"Failed to generate for problem {problem['problem_id']}")
            with open(args.save_path, 'a') as file:
                json.dump(
                    {
                        "success": False,
                        "task_id": problem["problem_id"], 
                        "image_id": img_id + 1,
                        "image_hash": hashnum,
                        "fail_reason": "No response"
                    }, 
                    file  # No indents!
                )
                file.write('\n')  # Add a newline for separation between entries
            break  # Break out of the run_id loop to move on to the next problem
        
        relavance_json = extract_result(response)
        if not relavance_json:
            print(f"Failed to extract result for problem {problem['problem_id']}")
            with open(args.save_path, 'a') as file:
                json.dump(
                    {
                        "success": False,
                        "task_id": problem["problem_id"], 
                        "image_id": img_id + 1,
                        "image_hash": hashnum,
                        "output": response,
                        "fail_reason": "No json"
                    }, 
                    file  # No indents!
                )
                file.write('\n')  # Add a newline for separation between entries
            break


        relavance = json.loads(relavance_json)["relevance"]

        with open(args.save_path, 'a') as file:
            json.dump(
                {
                    "success": True,
                    "task_id": problem["problem_id"], 
                    "image_id": img_id + 1,
                    "image_hash": hashnum,
                    "output": response,
                    "relavance": relavance
                }, 
                file  # No indents!
            )
            file.write('\n')  # Add a newline for separation between entries
            